# PDF Processing via LLMaaS

## 0. Setting your Name and Email

Please starting by putting your name and email in the following variables - please stick to the required format i.e. NAME_SURNAME

In [4]:
# WRITE YOUR NAME_SURNAME HERE, AS WELL AS YOUR EMAIL WITH WHICH YOU LOGGED IN INTO CELONIS
MY_NAME = 'SACHIN_DESAI'
MY_EMAIL = 'sachin1304desai@gmail.com'

## 1. Extract information from Invoices

This is the section you will need to fill in. Your code should create the following
- **a pandas dataframe called df that includes the extracted information**.
- **the dataframe should contain a column called 'po_reference' that contains the reference to the PO**
- **the values in the column 'po_reference' should be a 11-char long strings. Use left padding with zeros where needed.**

In [ ]:
# %pip install -q anthropic pydantic pymupdf pillow pandas

# ============================================================
# Invoice extraction with Claude (vision)
# Each invoice page is sent to Claude as an image; Claude returns the fields as
# schema-validated JSON. Works for any invoice layout - no OCR or per-layout rules.
# Output: pandas DataFrame `df` with the 7 required columns.
# Auth:   set the ANTHROPIC_API_KEY environment variable (never hard-code the key here).
# ============================================================
import base64, glob, io, os, re
from typing import Optional

import anthropic
import pandas as pd
import pymupdf                      # renders PDFs to images
from PIL import Image
from pydantic import BaseModel, Field

INVOICE_DIR = "Invoices"            # folder with the PDFs / images
MODEL = "claude-opus-5-5"

COLUMNS = ["vendor_name", "vendor_address", "payment_terms", "invoice_value",
           "company_code", "po_reference", "invoice_id"]

client = anthropic.Anthropic()      # reads ANTHROPIC_API_KEY from the environment


# ---------- 1. Load any invoice (PDF or image) as PNG page images ----------
def load_pages(path, dpi=150):
    if path.lower().endswith(".pdf"):
        return [p.get_pixmap(dpi=dpi).tobytes("png") for p in pymupdf.open(path)]
    buf = io.BytesIO()
    Image.open(path).convert("RGB").save(buf, format="PNG")
    return [buf.getvalue()]


# ---------- 2. Ask Claude to read the invoice ----------
class Invoice(BaseModel):
    vendor_name: Optional[str] = Field(description="Supplier/seller issuing the invoice (not the bill-to party)")
    vendor_address: Optional[str] = Field(description="Supplier's full postal address on one line, "
                                                      "comma-separated, without phone or email")
    payment_terms: Optional[int] = Field(description="Payment terms in days, e.g. 'Net 30' -> 30. If not stated, "
                                                     "the number of days between invoice date and due date")
    invoice_value: Optional[float] = Field(description="Total invoice amount including VAT/tax and all charges "
                                                       "(the final total, not the subtotal)")
    company_code: Optional[str] = Field(description="Bill-to company code, e.g. 'CompanyCode3'")
    po_reference: Optional[str] = Field(description="Purchase order reference, digits only, exactly as printed")
    invoice_id: Optional[str] = Field(description="Invoice number, e.g. 'INV-2021-001'")

PROMPT = ("Extract the fields from this supplier invoice. Read values exactly as printed. "
          "Use null for anything that is not on the invoice.")

def extract_claude(pages):
    content = [{"type": "image",
                "source": {"type": "base64", "media_type": "image/png",
                           "data": base64.standard_b64encode(png).decode()}} for png in pages]
    content.append({"type": "text", "text": PROMPT})
    response = client.beta.messages.parse(
        model=MODEL,
        max_tokens=16000,
        output_config={"effort": "medium"},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",          # re-run on a fallback model if a request is declined
        messages=[{"role": "user", "content": content}],
        output_format=Invoice,
    )
    if response.stop_reason == "refusal" or response.parsed_output is None:
        raise RuntimeError(f"No extraction (stop_reason={response.stop_reason})")
    return response.parsed_output.model_dump()


# ---------- 3. Normalise to clean values ----------
def normalise(rec):
    po = re.sub(r"\D", "", str(rec.get("po_reference") or ""))
    return {
        "vendor_name": (rec.get("vendor_name") or "").strip() or None,
        "vendor_address": re.sub(r"\s+", " ", rec.get("vendor_address") or "").strip(" ,") or None,
        "payment_terms": rec.get("payment_terms"),                 # days, e.g. 30
        "invoice_value": rec.get("invoice_value"),                 # gross, incl. VAT
        "company_code": (rec.get("company_code") or "").strip() or None,
        "po_reference": po.zfill(11) if po else None,              # 11 chars, left-padded
        "invoice_id": (rec.get("invoice_id") or "").strip() or None,
    }


# ---------- 4. Run over every invoice in the folder ----------
records = []
files = sorted(f for f in glob.glob(os.path.join(INVOICE_DIR, "*"))
               if f.lower().endswith((".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff")))
for path in files:
    print(f"Extracting {os.path.basename(path).strip()}")
    records.append(normalise(extract_claude(load_pages(path))))

df = pd.DataFrame(records, columns=COLUMNS)
df.to_csv("invoice_extraction.csv", index=False)

# ---------- 5. Quality checks ----------
assert df["po_reference"].str.len().eq(11).all(), "po_reference must be 11 characters"
missing = df[df.isna().any(axis=1)]
if not missing.empty:
    print("Rows with missing fields - please review:\n", missing)
df

#import base64, glob, io, os
#import pandas as pd
#import pymupdf
# from PIL import Image

# """ INVOICE_DIR = "Invoices"

# def to_data_uri(path, width=600, quality=55):
#     """First page of the invoice as a compressed JPEG, encoded as a base64 data URI."""
#     if path.lower().endswith(".pdf"):
#         page = pymupdf.open(path)[0]
#         img = Image.open(io.BytesIO(page.get_pixmap(dpi=120).tobytes("png")))
#     else:
#         img = Image.open(path)
#     img = img.convert("RGB")
#     if img.width > width:
#         img = img.resize((width, int(img.height * width / img.width)), Image.LANCZOS)
#     buf = io.BytesIO()
#     img.save(buf, "JPEG", quality=quality, optimize=True)
#     return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

# # map each invoice file to its image, keyed by file name (e.g. "INV-2021-001")
# images = {}
# for f in glob.glob(os.path.join(INVOICE_DIR, "*")):
#     if f.lower().endswith((".pdf", ".png", ".jpg", ".jpeg")):
#         images[os.path.splitext(os.path.basename(f))[0].strip()] = to_data_uri(f)

# # attach to df by invoice_id, so the order of rows doesn't matter
# df["invoice_image"] = df["invoice_id"].str.strip().map(images)

# print(df[["invoice_id", "po_reference"]].assign(image_chars=df["invoice_image"].str.len()))
# assert df["invoice_image"].notna().all(), "Some invoices have no image - check file names vs invoice_id" """

Extracting INV-2020-08-001247.pdf
Extracting INV-2020-001.png
Extracting INV-2020-07-001853.pdf
Extracting INV-2021-001.png
Extracting INV-2021-002.png
           invoice_id po_reference  image_chars
0  INV-2020-08-001247  00000586652        41491
1        INV-2020-001  00000048334        53243
2  INV-2020-07-001853  00001135572        64595
3        INV-2021-001  00000000181        52059
4        INV-2021-002  00000000191        49303


## 2. Pushing Data back to Celonis

The below script will push your results into Celonis, it may take **up to 5 minutes** for the results to show up in Celonis frontend. Please refrain from triggering the below script multiple times in a row.

In [6]:
%run push.ipynb

Successfully pushed file to Celonis
